In [1]:
import pandas as pd
import numpy as np

df = pd.read_excel("C:/Users/admin/Downloads/ashapura_mulund2.xlsx", header=8)
df.head()

,PROJECT_NAME,TIMESTAMP,PM2.5,PM10,TEMPERATURE,HUMIDITY
0,Ashapura Developers,2025-07-01T14:02:01+05:30,16,21,25.8,81.5
1,Ashapura Developers,2025-07-01T14:02:32+05:30,16,17,25.8,80.5
2,Ashapura Developers,2025-07-01T14:03:02+05:30,16,20,25.8,80.5
3,Ashapura Developers,2025-07-01T14:03:32+05:30,16,20,25.8,80.5
4,Ashapura Developers,2025-07-08T10:02:11+05:30,17,25,28.9,94.1


In [2]:
#Converting from timestamp to datetime
df['TIMESTAMP'] = pd.to_datetime(df['TIMESTAMP'], format='mixed')

In [3]:
#Set Index
df = df.set_index('TIMESTAMP')

In [4]:
# Sort properly
df = df.sort_values('TIMESTAMP')

In [5]:
# Step 4: Remove unrealistic values

df_clean = df.copy()

# Set extreme values to NaN
df_clean.loc[df_clean['PM2.5'] > 500, 'PM2.5'] = np.nan
df_clean.loc[df_clean['PM10'] > 600, 'PM10'] = np.nan

df_clean[['PM2.5','PM10']].describe()

,PM2.5,PM10
count,508402.00000,508374.000000
mean,31.49097,47.946648
std,13.07812,19.916520
min,9.00000,13.000000
25%,20.00000,31.000000
50%,32.00000,46.000000
75%,42.00000,63.000000
max,262.00000,593.000000


In [6]:
# Step 5: Fill missing values smoothly

df_clean['PM2.5'] = df_clean['PM2.5'].interpolate(method='time')
df_clean['PM10'] = df_clean['PM10'].interpolate(method='time')

In [7]:
# Step 6: Apply rolling mean smoothing

df_clean['PM2.5'] = df_clean['PM2.5'].rolling(window=3, min_periods=1).mean()
df_clean['PM10'] = df_clean['PM10'].rolling(window=3, min_periods=1).mean()

In [8]:
# Step 7: Clip to safe realistic ranges

df_clean['PM2.5'] = df_clean['PM2.5'].clip(0, 300)
df_clean['PM10'] = df_clean['PM10'].clip(0, 400)

In [9]:
#Round-Of Data
df_clean[['PM2.5','PM10','TEMPERATURE','HUMIDITY']] = df_clean[['PM2.5','PM10','TEMPERATURE','HUMIDITY']].round(2)

In [10]:
sensor_df = df_clean.copy()

In [11]:
#Resample data into 1hour batch
sensor_df = df.resample('1h').mean(numeric_only=True)

In [12]:
#Checking  Type of Index
print(type(df.index))

<class 'pandas.core.indexes.datetimes.DatetimeIndex'>


In [13]:
#Interpolate: to replace with mode value of any  columns
sensor_df = sensor_df.interpolate(limit_direction='forward')

In [14]:
#Reset Index
sensor_df = sensor_df.reset_index()

In [17]:
sensor_df.head()

,TIMESTAMP,PM2.5,PM10,TEMPERATURE,HUMIDITY
0,2025-07-01 14:00:00+05:30,16.00,19.50,25.80,80.75
1,2025-07-01 15:00:00+05:30,16.01,19.55,25.82,80.83
2,2025-07-01 16:00:00+05:30,16.02,19.60,25.84,80.90
3,2025-07-01 17:00:00+05:30,16.03,19.65,25.86,80.98
4,2025-07-01 18:00:00+05:30,16.04,19.70,25.88,81.06


In [16]:
#Round-Of Data
sensor_df[['PM2.5','PM10','TEMPERATURE','HUMIDITY']] = sensor_df[['PM2.5','PM10','TEMPERATURE','HUMIDITY']].round(2)

In [20]:
sensor_df.size

32875

Weather Data

In [34]:
#Weather Data 
weather_df = pd.read_excel("C:/Users/admin/Downloads/near_by_mulund_west.xlsx", header=16)
weather_df

,From Date,To Date,NO2,Ozone,WS,WD,SR
0,01-07-2025 00:00,01-07-2025 01:00,8.76,20.17,0.72,155.24,NaN
1,01-07-2025 01:00,01-07-2025 02:00,6.93,20.22,0.74,148.02,NaN
2,01-07-2025 02:00,01-07-2025 03:00,6.98,19.38,0.70,141.14,NaN
3,01-07-2025 03:00,01-07-2025 04:00,9.23,19.18,0.76,153.36,NaN
4,01-07-2025 04:00,01-07-2025 05:00,7.88,20.18,0.62,144.02,NaN
...,...,...,...,...,...,...,...
6587,01-04-2026 11:00,01-04-2026 12:00,9.16,11.41,NaN,241.53,128.97
6588,01-04-2026 12:00,01-04-2026 13:00,9.09,16.88,NaN,314.40,312.48
6589,01-04-2026 13:00,01-04-2026 14:00,9.03,25.56,NaN,95.41,328.68
6590,01-04-2026 14:00,01-04-2026 15:00,8.98,24.72,NaN,347.05,92.87


In [35]:
weather_df.isnull().sum()

From Date       0
To Date         0
NO2          2191
Ozone        1720
WS           5370
WD           3525
SR           5295
dtype: int64

In [36]:
#Remode To Date Columns
weather_df = weather_df.drop('To Date',axis=1)

In [37]:
weather_df.set_index('From Date')

,NO2,Ozone,WS,WD,SR
From Date,,,,,
01-07-2025 00:00,8.76,20.17,0.72,155.24,NaN
01-07-2025 01:00,6.93,20.22,0.74,148.02,NaN
01-07-2025 02:00,6.98,19.38,0.70,141.14,NaN
01-07-2025 03:00,9.23,19.18,0.76,153.36,NaN
01-07-2025 04:00,7.88,20.18,0.62,144.02,NaN
...,...,...,...,...,...
01-04-2026 11:00,9.16,11.41,NaN,241.53,128.97
01-04-2026 12:00,9.09,16.88,NaN,314.40,312.48
01-04-2026 13:00,9.03,25.56,NaN,95.41,328.68


In [38]:
#Replace From Date to TIMESTAMP
weather_df.rename(columns={'From Date':'TIMESTAMP'}, inplace=True)

In [39]:
weather_df = weather_df.set_index('TIMESTAMP')

In [40]:
#Making Timestamp same
sensor_df['TIMESTAMP'] = sensor_df['TIMESTAMP'].dt.tz_localize(None)
#print(sensor_df.columns)
#sensor_df.head()

In [41]:
print(weather_df.columns)

Index(['NO2', 'Ozone', 'WS', 'WD', 'SR'], dtype='object')


In [42]:
weather_df = weather_df.reset_index()

In [43]:
weather_df['TIMESTAMP'] = pd.to_datetime(
    weather_df['TIMESTAMP'],
    format='%d-%m-%Y %H:%M'
)

In [44]:
weather_df.head()

,TIMESTAMP,NO2,Ozone,WS,WD,SR
0,2025-07-01 00:00:00,8.76,20.17,0.72,155.24,NaN
1,2025-07-01 01:00:00,6.93,20.22,0.74,148.02,NaN
2,2025-07-01 02:00:00,6.98,19.38,0.70,141.14,NaN
3,2025-07-01 03:00:00,9.23,19.18,0.76,153.36,NaN
4,2025-07-01 04:00:00,7.88,20.18,0.62,144.02,NaN


In [45]:
weather_df = weather_df.dropna()

In [46]:
weather_df.head()

,TIMESTAMP,NO2,Ozone,WS,WD,SR
8,2025-07-01 08:00:00,11.57,19.25,0.70,153.08,14.20
9,2025-07-01 09:00:00,15.18,18.69,0.63,134.70,80.93
10,2025-07-01 10:00:00,14.57,20.83,0.72,152.81,99.88
11,2025-07-01 11:00:00,13.88,21.47,0.77,159.03,133.20
12,2025-07-01 12:00:00,16.81,20.84,0.86,170.73,67.17


In [48]:
weather_df.shape

(688, 6)

In [78]:
#Merge Data So our ML model can train
merged_df1 = pd.merge(sensor_df,weather_df, on="TIMESTAMP")

In [82]:
merged_df1.to_csv("model_data.csv")

In [49]:
#Merge Data So our ML model can train
merged_df = pd.merge(sensor_df,weather_df, on="TIMESTAMP")

In [50]:
merged_df = merged_df.dropna()

In [51]:
merged_df.size

6820

In [77]:
print(merged_df.columns)

Index(['PM2.5', 'PM10', 'TEMPERATURE', 'HUMIDITY', 'NO2', 'Ozone', 'WS', 'WD',
       'SR', 'hour', 'day', 'month', 'hour_sin', 'hour_cos', 'PM25_lag_1',
       'PM10_lag_1', 'PM25_lag_2', 'PM10_lag_2', 'PM25_lag_3', 'PM10_lag_3',
       'PM25_lag_6', 'PM10_lag_6', 'PM25_lag_12', 'PM10_lag_12', 'PM25_lag_24',
       'PM10_lag_24', 'PM25_lag_48', 'PM10_lag_48', 'PM25_lag_72',
       'PM10_lag_72', 'PM25_roll_mean_6', 'PM10_roll_mean_6',
       'PM25_roll_mean_12', 'PM10_roll_mean_12', 'PM25_roll_mean_24',
       'PM10_roll_mean_24', 'PM25_diff', 'PM10_diff'],
      dtype='object')


In [53]:
merged_df['TIMESTAMP'] = pd.to_datetime(merged_df['TIMESTAMP'], errors='coerce')

merged_df = merged_df.sort_values('TIMESTAMP')

merged_df.set_index('TIMESTAMP', inplace=True)

# FORCE datetime index
merged_df.index = pd.to_datetime(merged_df.index)

In [54]:
merged_df["hour"] = merged_df.index.hour
merged_df["day"] = merged_df.index.day
merged_df["month"] = merged_df.index.month

# Cyclical encoding (better for ML)
merged_df["hour_sin"] = np.sin(2 * np.pi * merged_df["hour"] / 24)
merged_df["hour_cos"] = np.cos(2 * np.pi * merged_df["hour"] / 24)

In [55]:
lags = [1, 2, 3, 6, 12, 24, 48, 72]

for lag in lags:
    merged_df[f"PM25_lag_{lag}"] = merged_df["PM2.5"].shift(lag)
    merged_df[f"PM10_lag_{lag}"] = merged_df["PM10"].shift(lag)

In [56]:
windows = [6, 12, 24]

for w in windows:
    merged_df[f"PM25_roll_mean_{w}"] = merged_df["PM2.5"].rolling(w).mean()
    merged_df[f"PM10_roll_mean_{w}"] = merged_df["PM10"].rolling(w).mean()

In [57]:
merged_df["PM25_diff"] = merged_df["PM2.5"].diff()
merged_df["PM10_diff"] = merged_df["PM10"].diff()

In [58]:
merged_df = merged_df.dropna()
print("After feature engineering:", merged_df.shape)

After feature engineering: (610, 38)


In [69]:
target_cols = ["PM2.5", "PM10"]

feature_cols = [
    'PM2.5', 'PM10', 'TEMPERATURE', 'HUMIDITY',
    'WS', 'WD', 'Ozone', 'NO2', 'SR'
]

X = merged_df[feature_cols]
y = merged_df[target_cols]

Prepare Last 7 days

In [70]:
WINDOW = 168  # 7 days

last_7_days = X.tail(WINDOW)

if len(last_7_days) < WINDOW:
    raise ValueError("Not enough data")

X_input = last_7_days.values.reshape(1, WINDOW, len(feature_cols))

In [66]:
from tensorflow.keras.models import load_model

model = load_model("lstm_model.h5", compile=False)

In [68]:
merged_df.head()

,PM2.5,PM10,TEMPERATURE,HUMIDITY,NO2,Ozone,WS,WD,SR,hour,...,PM25_lag_72,PM10_lag_72,PM25_roll_mean_6,PM10_roll_mean_6,PM25_roll_mean_12,PM10_roll_mean_12,PM25_roll_mean_24,PM10_roll_mean_24,PM25_diff,PM10_diff
TIMESTAMP,,,,,,,,,,,,,,,,,,,,,
2025-07-20 15:00:00,18.13,27.03,28.09,98.33,16.79,16.73,1.95,173.09,128.00,15,...,16.00,19.50,17.880000,28.421667,18.432500,30.542500,18.432500,30.437500,-0.20,-2.38
2025-07-20 16:00:00,18.19,28.50,27.87,97.31,16.29,17.66,1.96,172.63,113.53,16,...,16.01,19.55,18.053333,28.700000,18.457500,30.509167,18.352083,30.154167,0.06,1.47
2025-07-20 17:00:00,18.26,29.97,27.65,96.29,15.98,17.76,1.98,172.90,17.50,17,...,16.02,19.60,18.373333,29.406667,18.495000,30.549167,18.286250,29.922083,0.07,1.47
2025-07-21 08:00:00,19.68,30.13,25.91,100.00,19.60,14.72,2.14,179.93,7.90,8,...,16.17,20.38,18.710000,29.610000,18.555000,30.260000,18.333750,29.950000,1.42,0.16
2025-07-21 09:00:00,19.34,28.41,26.11,100.00,18.47,15.14,2.13,179.85,62.55,9,...,16.18,20.43,18.655000,28.908333,18.484167,29.402500,18.382500,29.871667,-0.34,-1.72


Predict Next 7 days

In [74]:
def predict_7_days(model, last_window, steps=168):
    preds = []

    current = last_window.copy()

    for _ in range(steps):
        pred = model.predict(current)[0]  # shape (24,2)

        next_val = pred[0]  # take 1st hour prediction

        preds.append(next_val)

        # shift window
        current = np.roll(current, -1, axis=1)

        # update only PM2.5 & PM10
        current[0, -1, 0] = next_val[0]
        current[0, -1, 1] = next_val[1]

    return np.array(preds)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 99ms/step


IndexError: invalid index to scalar variable.

In [73]:
print(future.shape)

(1, 48)


In [72]:
start_time = datetime.now().replace(minute=0, second=0, microsecond=0)

time_index = [
    start_time + timedelta(hours=i)
    for i in range(168)
]

NameError: name 'datetime' is not defined

In [ ]:
future_df = pd.DataFrame({
    "Time": time_index,
    "PM2.5": pm25_pred,
    "PM10": pm10_pred
})

print(future_df.head())

Cause Detection

In [ ]:
def detect_cause(df):
    WINDOW = 168

    if len(df) < WINDOW * 2:
        return ["Not enough data"]

    prev = df.iloc[-WINDOW*2:-WINDOW]
    curr = df.iloc[-WINDOW:]

    causes = []

    def check(feature, name):
        prev_avg = prev[feature].mean()
        curr_avg = curr[feature].mean()

        if prev_avg == 0:
            return

        change = ((curr_avg - prev_avg) / prev_avg) * 100

        if change > 10:
            causes.append(f"⬆️ {name} increased by {round(change,2)}%")
        elif change < -10:
            causes.append(f"⬇️ {name} decreased by {round(change,2)}%")

    # Weather drivers
    check("TEMPERATURE", "Temperature")
    check("HUMIDITY", "Humidity")
    check("WS", "Wind Speed")
    check("Ozone", "Ozone")
    check("NO2", "NO2")

    # Pollution itself
    check("PM2.5", "PM2.5")
    check("PM10", "PM10")

    return causes

In [ ]:
causes = detect_cause(df)

print("\n🔍 Cause of Change:")
for c in causes:
    print(c)

In [ ]:
Smart Interpretation

In [ ]:
def summary(causes):
    insights = []

    if any("Wind Speed decreased" in c for c in causes):
        insights.append("Low wind → Pollution accumulation")

    if any("Temperature increased" in c for c in causes):
        insights.append("High temperature → Pollution reactions increase")

    if any("Ozone increased" in c for c in causes):
        insights.append("Ozone rise → Air quality worsening")

    return insights


insights = summary(causes)

print("\n🧾 Final Insights:")
for i in insights:
    print("⚠️", i)